In [ ]:
import sys, importlib, os

_src_training = os.path.dirname(os.path.abspath("training.ipynb"))
_src = os.path.dirname(_src_training)
for _p in [_src_training, _src]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import infrastructure, pipeline, classifier, evaluation
for _mod in [infrastructure, pipeline, classifier, evaluation]:
    importlib.reload(_mod)

from infrastructure import clean_neo4j_db, clean_kafka_topics, delete_test_neo4j_nodes, verify_concept_created, reload_concept_cache
from pipeline import train_mnist, remove_concept, retrain_concept
from evaluation import test_mnist_all
from classifier import classify_image
import json, uuid
print("Modules loaded.")

In [ ]:
classes_to_subclasses = {
    0: [1],
    1: [1, 3],
    2: [1, 2],
    3: [1],
    4: [1, 2],
    5: [1],
    6: [1],
    7: [1],
    8: [1],
    9: [2],
}

In [ ]:
clean_neo4j_db()

for class_num in classes_to_subclasses:
    for subclass in classes_to_subclasses[class_num]:
        train_mnist(class_number=class_num, subclass=subclass, is_prepared_samples=True, with_concept_creation=True)

In [ ]:
params = {
    "ged_timeout": 5,
    "skeletonization_threshold": 110,
    "simplification_epsilon": 4.55,
    "diagnostic_weight_epsilon": 0.1,
    "delete_image_nodes": False,
    "node_costs": {
        "NO_COST": 0.0,
        "MINOR": 0.25,
        "GENERAL": 0.5,
        "SEVERE": 0.8,
        "NO_MATCH": 1,
        "IMPOSSIBLE": 6,
    },
    "features": [
        "normalized_x", "normalized_y",
        "distance_to_centroid",
        "horizontal_direction", "vertical_direction", "angle_with_ox",
        "junction_angle_min",
        "is_endpoint", "is_corner",
        "length_ratio_to_max",
        # "eccentricity",
        "avg_neighbor_vector_length",
        "neighbor_endpoint_count", "neighbor_junction_count",
    ],
}
# Pick up any retrained concepts on the running classification instances (no redeploy).
reload_concept_cache()
results, y_true, y_pred, run_dir = test_mnist_all(
    classes=list(classes_to_subclasses.keys()),
    params=params,
    sample_fraction=1.0,
    description="Retrained concepts with the diameter instead of distance to centroid.",
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

delete_test_neo4j_nodes()

class_number = 8
img_num = 883
image_id = f"mnist_test_{class_number}_{img_num:05d}"
local_path = f"../../datasets/mnist_all/{class_number}"
nuclio_path = f"/opt/nuclio/shared_storage/mnist_all/{class_number}"

img_file = f"{image_id}.png"
img_local = os.path.join(local_path, img_file)
if os.path.exists(img_local):
    img = mpimg.imread(img_local)
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    plt.title(f"MNIST Class {class_number}, Image #{img_num}")
    plt.axis("off")
    plt.show()

image_id_for_test = str(uuid.uuid4())
params = {
    "ged_timeout": 5,
    "skeletonization_threshold": 110,
    "image_id": image_id_for_test,
    "session_id": "test",
    "delete_image_nodes": False,
    # "simplification_epsilon": 5,
}
result = classify_image(os.path.join(nuclio_path, img_file), params=params, timeout=60)
print(json.dumps(result, indent=2))

In [ ]:
# remove_concept("3_1")
retrain_concept(number=4, subclass=1, with_concept_creation=True)